In [1]:
# ╔════════════════════════════════════════════════════════════════════╗
# ║  One row per *predicted exon* – updated for (5, N) predictions     ║
# ╚════════════════════════════════════════════════════════════════════╝
from __future__ import annotations
import h5py, numpy as np, pandas as pd
from pathlib import Path
from typing import List, Tuple, Dict

# ──────────────────────────────────────────────────────────────────────
# ① Poison exon list + gene‑name → Ensembl map
# ──────────────────────────────────────────────────────────────────────
def read_poison_exons(path: str | Path,
                      keep_chrs: set[str] = {"chr8", "chr20", "chr21"}) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t", header=None, dtype=str)
    coords = df[0].str.split("_", expand=True)
    coords.columns = ["chrom", "start", "end", "strand"]
    out = pd.concat([coords, df[[1]]], axis=1)
    out.columns = ["chrom", "start", "end", "strand", "gene_symbol"]
    out = out[out.chrom.isin(keep_chrs)].copy()
    out[["start", "end"]] = out[["start", "end"]].astype(int)
    out["poison_idx"] = out.index                  # unique id
    return out

def read_gene_map(path: str | Path) -> pd.DataFrame:
    g = pd.read_csv(path, dtype=str)
    g["ensembl_gene_id"] = g["ID"].str.split(".").str[0]
    return g[["ensembl_gene_id", "gene_name"]].drop_duplicates()

# ──────────────────────────────────────────────────────────────────────
# ② Predictions → exon segments
# ──────────────────────────────────────────────────────────────────────
def load_avg_predictions(grp) -> np.ndarray:
    """
    Returns averaged predictions with shape (seq_len, 5):
      • source datasets are (5, seq_len)
      • we average forward & reverse then transpose
    """
    fwd = grp["gene_predictions_forward"][:]    # (5, N)
    rev = grp["gene_predictions_reverse"][:]    # (5, N)
    return (0.5 * (fwd + rev[:, ::-1])).T                # (N, 5)

def exon_calls_from_probs(p: np.ndarray,
                          intron_col=2, exon_col=1) -> np.ndarray:
    """Arg‑max only between INTRON (2) and EXON (1) columns."""
    mask = np.argmax(p[:, [intron_col, exon_col]], axis=1)
    return mask.astype(bool)                    # True = exon

def contiguous_segments(mask: np.ndarray) -> List[Tuple[int,int]]:
    if mask.sum() == 0: return []
    diff   = np.diff(mask.astype(int))
    starts = np.flatnonzero(diff == 1) + 1
    ends   = np.flatnonzero(diff == -1) + 1
    if mask[0]:  starts = np.r_[0, starts]
    if mask[-1]: ends   = np.r_[ends, mask.size]
    return list(zip(starts, ends))              # (s, e) end‑exclusive

def idx_to_genomic(segs: List[Tuple[int,int]],
                   tx_start: int,
                   coord_base: int = 0) -> List[Tuple[int,int]]:
    """
    Convert transcript‑relative indices to genomic coordinates.
    • All input files use the *positive‑strand convention* (start < end).
    • For both real +/– strands we do the same simple formula:
        genomic_start = tx_start + s
        genomic_end   = tx_start + e
      because coordinates already grow left‑to‑right.
    • If coord_base == 1 → make the interval 1‑based closed.
    """
    out = []
    for s, e in segs:
        g0 = tx_start + s
        g1 = tx_start + e                     # still end‑exclusive
        if coord_base == 1:                  # 1‑based CLOSED
            g0 += 1
        out.append((int(g0), int(g1)))
    return out

# ──────────────────────────────────────────────────────────────────────
# ③ Walk HDF5 files and collect *all* predicted exons for target genes
# ──────────────────────────────────────────────────────────────────────
def _walk(h):
    yield h
    for k, v in h.items():
        if isinstance(v, h5py.Group):
            yield from _walk(v)

def collect_predicted_exons(h5_files: List[str | Path],
                            genes_of_interest: set[str],
                            coord_base: int = 0) -> pd.DataFrame:
    rows: List[Dict] = []
    for path in h5_files:
        with h5py.File(path, "r") as hf:
            for grp in _walk(hf):
                # recognise transcript leaf groups
                if {"gene_predictions_forward",
                    "gene_predictions_reverse",
                    "gene_coordinates"}.issubset(grp.keys()) and "ID" in grp.attrs:
                    
                    gid = grp.attrs["ID"]
                    gid = gid.decode() if isinstance(gid, bytes) else gid
                    gid = gid.split(".")[0]
                    if gid not in genes_of_interest:
                        continue

                    chrom = grp.attrs["chromosome"]
                    chrom = chrom.decode() if isinstance(chrom, bytes) else chrom

                    coords = np.asarray(grp["gene_coordinates"][:]).flatten()
                    if coords.size != 2:
                        raise ValueError(f"gene_coordinates expected length 2, got {coords.shape}")
                    tx_start, tx_end = int(coords[0]), int(coords[1])  # tx_end unused now

                    # ---- derive exon segments ---------------------------------
                    preds = load_avg_predictions(grp)                 # (N, 5)
                    mask  = exon_calls_from_probs(preds)
                    segs  = contiguous_segments(mask)
                    gsegs = idx_to_genomic(segs, tx_start, coord_base)

                    for g0, g1 in gsegs:
                        rows.append({"ensembl_gene_id": gid,
                                      "chrom": chrom,
                                      "pred_start": g0,
                                      "pred_end":   g1})
    df = pd.DataFrame(rows)
    df["pred_idx"] = df.index                 # unique id per predicted exon
    return df

# ──────────────────────────────────────────────────────────────────────
# ④ Compare each predicted exon against poison exons of same gene+chr
# ──────────────────────────────────────────────────────────────────────
_SCORE = {"none": 0, "partial": 1, "full": 2}

def _overlap(a0,a1,b0,b1):
    if b1 < a0 or b0 > a1:      return "none"
    if b0 <= a0 and b1 >= a1:   return "full"
    return "partial"

def evaluate_overlaps(pred_df, poison_df, coord_base=0):
    merged = pred_df.merge(poison_df,
                           on=["ensembl_gene_id", "chrom"],
                           how="left",
                           suffixes=("_pred","_poison"))

    def classify(r):
        if pd.isna(r["start"]):            # gene has no poison exon on this chrom
            return "none"
        a0 = r["pred_start"]
        a1 = r["pred_end"]   if coord_base==1 else r["pred_end"]-1
        b0 = r["start"]
        b1 = r["end"]        if coord_base==1 else r["end"]-1
        return _overlap(a0,a1,b0,b1)

    merged["overlap"] = merged.apply(classify, axis=1)
    merged["score"]   = merged["overlap"].map(_SCORE)

    best = (merged.sort_values(["pred_idx","score"], ascending=[True,False])
                   .drop_duplicates("pred_idx")
                   .drop(columns="score"))

    return (best.rename(columns={"start":"poison_start",
                                 "end":"poison_end"})
                [["ensembl_gene_id","chrom",
                  "pred_start","pred_end",
                  "poison_start","poison_end",
                  "overlap"]]
            .reset_index(drop=True))

# ──────────────────────────────────────────────────────────────────────
# ⑤ End‑to‑end wrapper
# ──────────────────────────────────────────────────────────────────────
def run_pipeline(POISON_TSV, GENE_MAP_CSV, PRED_FILES, COORD_BASE=0):
    # 1. Load ground‑truth poison exons
    poison = read_poison_exons(POISON_TSV)
    gmap   = read_gene_map(GENE_MAP_CSV)
    poison = (poison.merge(gmap,
                           left_on="gene_symbol",
                           right_on="gene_name",
                           how="left")
                    .dropna(subset=["ensembl_gene_id"]))

    # 2. Collect all predicted exons for those genes
    genes_of_interest = set(poison["ensembl_gene_id"])
    pred_exons = collect_predicted_exons(PRED_FILES,
                                         genes_of_interest,
                                         coord_base=COORD_BASE)

    # 3. Evaluate overlaps (one row per predicted exon)
    overlaps = evaluate_overlaps(pred_exons, poison, coord_base=COORD_BASE)
    return overlaps

# ──────────────────────────────────────────────────────────────────────
# ⑥ User parameters – EDIT ME
# ──────────────────────────────────────────────────────────────────────
POISON_TSV   = "SupplementaryDataFile1 - ST1.tsv"
GENE_MAP_CSV = "Hs-hg38_genes_names.csv"
PRED_FILES   = ["/home/jovyan/shares/SR003.nfs2/caduseus_artem/neurips_rebbutal/pred/caduceus_ps-250k_shawerma-hg38-genes-Hs-chr20.hdf5",
                "/home/jovyan/dnalm/downstream_tasks/annotation/poison_exons/caduceus_ps_250k_shawerma_again_hg38_chr8_chr21_poison_Hs_chr21.hdf5"]
COORD_BASE   = 0        # 0 = BED half‑open; 1 = GTF closed
WRITE_CSV    = False
OUT_CSV      = "predicted_exon_vs_poison.csv"

# ──────────────────────────────────────────────────────────────────────
# ⑦ Run
# ──────────────────────────────────────────────────────────────────────
overlaps = run_pipeline(POISON_TSV, GENE_MAP_CSV, PRED_FILES, COORD_BASE)

print(f"✅ Evaluated {len(overlaps)} predicted exons "
      f"(one row per exon segment).")
display(overlaps.head())

if WRITE_CSV:
    overlaps.to_csv(OUT_CSV, index=False)
    print(f"📄 Saved to {OUT_CSV}")


✅ Evaluated 962 predicted exons (one row per exon segment).


,ensembl_gene_id,chrom,pred_start,pred_end,poison_start,poison_end,overlap
0,ENSG00000054803,chr20,55997356,55998754,56001290,56001368,none
1,ENSG00000054803,chr20,55998755,55998756,56001290,56001368,none
2,ENSG00000054803,chr20,55998758,55998760,56001290,56001368,none
3,ENSG00000054803,chr20,55998761,55998762,56001290,56001368,none
4,ENSG00000054803,chr20,55998767,55998768,56001290,56001368,none


In [18]:
np.unique(overlaps.overlap.to_numpy(), return_counts=True)

(array(['none', 'partial'], dtype=object), array([961,   1]))

In [13]:
overlaps.loc[overlaps.overlap == 'full']

,ensembl_gene_id,chrom,pred_start,pred_end,poison_start,poison_end,overlap
324,ENSG00000104517,chr8,102302307,102302308,102302271,102302481,full
325,ENSG00000104517,chr8,102302322,102302323,102302271,102302481,full
326,ENSG00000104517,chr8,102302325,102302327,102302271,102302481,full
327,ENSG00000104517,chr8,102302328,102302329,102302271,102302481,full
328,ENSG00000104517,chr8,102302330,102302332,102302271,102302481,full
329,ENSG00000104517,chr8,102302333,102302335,102302271,102302481,full
330,ENSG00000104517,chr8,102302351,102302353,102302271,102302481,full
331,ENSG00000104517,chr8,102302354,102302356,102302271,102302481,full
332,ENSG00000104517,chr8,102302369,102302376,102302271,102302481,full


In [3]:
overlaps.loc[overlaps.gene_symbol == 'KIZ']

,chrom,start,end,strand,gene_symbol,ensembl_gene_id,gene_name,pred_start,pred_end,overlap
0,chr20,21216214,21216370,+,KIZ,ENSG00000088970,KIZ,21125982,21126204,none
1,chr20,21216214,21216370,+,KIZ,ENSG00000088970,KIZ,21132096,21132159,none
2,chr20,21216214,21216370,+,KIZ,ENSG00000088970,KIZ,21136389,21136552,none
3,chr20,21216214,21216370,+,KIZ,ENSG00000088970,KIZ,21145564,21145654,none
4,chr20,21216214,21216370,+,KIZ,ENSG00000088970,KIZ,21161870,21162507,none
5,chr20,21216214,21216370,+,KIZ,ENSG00000088970,KIZ,21162849,21163159,none
6,chr20,21216214,21216370,+,KIZ,ENSG00000088970,KIZ,21205490,21205584,none
7,chr20,21216214,21216370,+,KIZ,ENSG00000088970,KIZ,21214534,21214700,none
8,chr20,21216214,21216370,+,KIZ,ENSG00000088970,KIZ,21215582,21215648,none
9,chr20,21216214,21216370,+,KIZ,ENSG00000088970,KIZ,21229010,21229115,none


In [16]:
import h5py, numpy as np, random, pprint, pathlib

# ══════════════════════════════════════════════════════════════════════
# user settings
H5_PATH      = "/home/jovyan/shares/SR003.nfs2/caduseus_artem/neurips_rebbutal/pred/caduceus_ps-250k_shawerma-hg38-genes-Hs-chr20.hdf5"
COORD_BASE   = 0     # 0 = BED half‑open, 1 = GTF closed
# ══════════════════════════════════════════════════════════════════════

def contiguous_segments(mask: np.ndarray):
    """Return (start, end) pairs (0‑based, end‑exclusive) for each True stretch."""
    if mask.sum() == 0:
        return []
    diff   = np.diff(mask.astype(int))
    starts = np.flatnonzero(diff == 1) + 1
    ends   = np.flatnonzero(diff == -1) + 1
    if mask[0]:  starts = np.r_[0, starts]
    if mask[-1]: ends   = np.r_[ends, mask.size]
    return list(zip(starts, ends))

def idx_to_genomic(segs, tx_start, coord_base=0):
    """Convert transcript indices to genomic coords (positive‑strand convention)."""
    out = []
    for s, e in segs:
        g0 = tx_start + s
        g1 = tx_start + e          # end‑exclusive
        if coord_base == 1:        # make closed interval
            g0 += 1
        out.append((int(g0), int(g1)))
    return out

def pick_random_transcript(h5_path: str | pathlib.Path, coord_base=0):
    with h5py.File(h5_path, "r") as hf:
        # collect every group that looks like a transcript
        candidates = []
        def walk(g):
            if {"gene_predictions_forward",
                "gene_predictions_reverse",
                "gene_coordinates"}.issubset(g.keys()) and "ID" in g.attrs:
                candidates.append(g.name)
            for k, v in g.items():
                if isinstance(v, h5py.Group):
                    walk(v)
        walk(hf)

        if not candidates:
            raise RuntimeError("No transcript groups found in the file.")

        grp_name = random.choice(candidates)
        grp      = hf[grp_name]

        # basic metadata
        gid   = grp.attrs["ID"]
        gid   = gid.decode() if isinstance(gid, bytes) else gid
        chrom = grp.attrs["chromosome"]
        chrom = chrom.decode() if isinstance(chrom, bytes) else chrom

        # coordinates (always start < end)
        tx_start, tx_end = map(int, grp["gene_coordinates"][:])

        # average predictions, transpose to (N,5)
        fwd = grp["gene_predictions_forward"][:]   # (5, N)
        rev = grp["gene_predictions_reverse"][:]   # (5, N)
        preds = (0.5 * (fwd + rev[:, ::-1])).T              # (N, 5)

        # exon mask: column 1 (EXON) vs column 2 (INTRON)
        mask = np.argmax(preds[:, [2, 1]], axis=1) == 1   # True ↔ exon
        segs = contiguous_segments(mask)
        gsegs = idx_to_genomic(segs, tx_start, coord_base)

    return {
        "gene_id": gid,
        "chromosome": chrom,
        "tx_span": (tx_start, tx_end),
        "predicted_exons": gsegs          # list of (start, end)
    }

result = pick_random_transcript(H5_PATH, coord_base=COORD_BASE)

print("\nRandom transcript:", result["gene_id"], "on", result["chromosome"])
print("Transcript span  :", result["tx_span"])
print("Predicted exons  :")
pprint.pp(result["predicted_exons"])



Random transcript: ENSG00000306145.1 on chr20
Transcript span  : (20003223, 20003934)
Predicted exons  :
[(20003223, 20003345), (20003812, 20003934)]


In [1]:
# # ╔════════════════════════════════════════════════════════════════════╗
# # ║  Poison‑exon overlap + per‑ID exon counts                          ║
# # ╚════════════════════════════════════════════════════════════════════╝
# import csv, h5py, numpy as np, pandas as pd, re
# from collections import defaultdict

# # ──────────────────────────────────────────────────────────────────────
# # ①  Paths / options
# # ──────────────────────────────────────────────────────────────────────
# POISON_TSV   = "SupplementaryDataFile1 - ST1.tsv"
# GENE_MAP_CSV = "Hs-hg38_genes_names.csv"

# PRED_FILES   = [
#     "/home/jovyan/shares/SR003.nfs2/caduseus_artem/neurips_rebbutal/pred/gena_best-1M-again-hg38_genome_poison-Hs.hdf5"
# ]

# # PRED_FILES   = [
# #     "/home/jovyan/shares/SR003.nfs2/caduseus_artem/neurips_rebbutal/pred/caduceus_ps-250k_shawerma-hg38-genes-Hs-chr20.hdf5",
# #     "/home/jovyan/dnalm/downstream_tasks/annotation/poison_exons/caduceus_ps_250k_shawerma_again_hg38_chr8_chr21_poison_Hs_chr21.hdf5"
# # ]
# # PRED_FILES   = [
# #     "/home/jovyan/dnalm/downstream_tasks/annotation/poison_exons/gena_best-1M-again-hg38_chr8_chr21_poison-Hs.hdf5",
# #     "/home/jovyan/shares/SR003.nfs2/caduseus_artem/neurips_rebbutal/pred/genatator_again-hg38-Hs_chr20.hdf5"
# # ]
# COORD_BASE   = 1              # 0 = BED half‑open ; 1 = GTF closed
# # KEEP_CHRS    = {"chr8", "chr20", "chr21"}
# KEEP_CHRS    = {f'chr{i}' for i in range(1, 24)}


# # ──────────────────────────────────────────────────────────────────────
# # ②  Utilities
# # ──────────────────────────────────────────────────────────────────────
# coord_re = re.compile(r"^(chr[\w]+)_(\d+)_(\d+)(?:_[\+\-])?$")

# def parse_coord(txt):
#     m = coord_re.match(txt)
#     return (m.group(1), int(m.group(2)), int(m.group(3))) if m else None

# def contiguous_ones(arr):
#     idx = np.where(arr == 1)[0]
#     if idx.size == 0: return []
#     split = np.where(np.diff(idx) > 1)[0] + 1
#     return [(seg[0], seg[-1] + 1) for seg in np.split(idx, split)]

# def overlap_kind(a0,a1,b0,b1):
#     if a1 <= b0 or a0 >= b1:            return "none"
#     if a0 <= b0 and a1 >= b1:           return "full"
#     return "partial"

# # ──────────────────────────────────────────────────────────────────────
# # ③  Gene‑symbol ↔ Ensembl map
# # ──────────────────────────────────────────────────────────────────────
# ensg2sym = {}
# with open(GENE_MAP_CSV) as f:
#     for row in csv.DictReader(f):
#         ensg2sym.setdefault(row["ID"].split(".")[0], row["gene_name"])

# # ──────────────────────────────────────────────────────────────────────
# # ④  Load poison exons  (keyed by Ensembl ID)
# # ──────────────────────────────────────────────────────────────────────
# gene2poison = defaultdict(list)
# with open(POISON_TSV) as f:
#     for cols in csv.reader(f, delimiter="\t"):
#         if len(cols) < 2: continue
#         parsed = parse_coord(cols[0])
#         if parsed is None: continue
#         chrom, start, end = parsed
#         if chrom not in KEEP_CHRS: continue
#         gene_sym = cols[1]
#         # try to get ENSG from the same row or mapping table
#         ensg = next((c.split(".")[0] for c in cols[2:] if c.startswith("ENSG")),
#                     None) or next((k for k,v in ensg2sym.items() if v == gene_sym), None)
#         if ensg:
#             gene2poison[ensg].append({"chrom": chrom, "start": start, "end": end, "symbol": gene_sym})
#             ensg2sym.setdefault(ensg, gene_sym)

# # ──────────────────────────────────────────────────────────────────────
# # ⑤  Scan HDF5 files → predicted exon segments
# # ──────────────────────────────────────────────────────────────────────
# def is_tx(g):
#     k = g.keys()
#     return {"gene_predictions_forward","gene_predictions_reverse","gene_coordinates"}.issubset(k) \
#            and "ID" in g.attrs

# def walk(g):
#     yield g
#     for v in g.values():
#         if isinstance(v, h5py.Group):
#             yield from walk(v)

# pred_rows = []
# exon_counts = defaultdict(int)          # NEW: exon count per Ensembl ID

# for path in PRED_FILES:
#     with h5py.File(path, "r") as hf:
#         for grp in walk(hf):
#             if not is_tx(grp): continue
#             gid = grp.attrs["ID"]
#             gid = (gid.decode() if isinstance(gid, bytes) else gid).split(".")[0]
#             if gid not in gene2poison:   continue

#             chrom = grp.attrs["chromosome"]
#             chrom = chrom.decode() if isinstance(chrom, bytes) else chrom
#             tx_start, tx_end = map(int, grp["gene_coordinates"][:])

#             avg = 0.5*(grp["gene_predictions_forward"][:] + grp["gene_predictions_reverse"][:])  # 5×N
#             exon_mask = np.argmax(avg[[2,1],:], axis=0) == 1
#             for s,e in contiguous_ones(exon_mask):
#                 exon_counts[gid] += 1                     # count for this ID
#                 g0, g1 = tx_start + s, tx_start + e
#                 if COORD_BASE == 1: g0 += 1
#                 pred_rows.append({"gene_id": gid, "chrom": chrom,
#                                   "pred_start": g0, "pred_end": g1})

# # ──────────────────────────────────────────────────────────────────────
# # ⑥  Attach gene_symbol + one poison exon per row
# # ──────────────────────────────────────────────────────────────────────
# def choose_poison(ensg, chrom, mid):
#     poisons = gene2poison[ensg]
#     same = [p for p in poisons if p["chrom"] == chrom] or poisons
#     return min(same, key=lambda p: abs((p["start"]+p["end"])/2 - mid))

# full_rows = []
# for r in pred_rows:
#     gid, chrom = r["gene_id"], r["chrom"]
#     mid = (r["pred_start"] + (r["pred_end"] if COORD_BASE else r["pred_end"]-1))/2
#     pe = choose_poison(gid, chrom, mid)
#     a0, a1 = r["pred_start"], r["pred_end"] if COORD_BASE else r["pred_end"]-1
#     b0, b1 = pe["start"], pe["end"] if COORD_BASE else pe["end"]-1
#     r.update({"gene_symbol": ensg2sym.get(gid),
#               "poison_start": pe["start"],
#               "poison_end":   pe["end"],
#               "overlap":      overlap_kind(a0,a1,b0,b1)})
#     full_rows.append(r)

# # ──────────────────────────────────────────────────────────────────────
# # ⑦  Final DataFrame and exon‑count print‑out
# # ──────────────────────────────────────────────────────────────────────
# results_df = pd.DataFrame(full_rows,
#     columns=["gene_id","gene_symbol","chrom","pred_start","pred_end",
#              "poison_start","poison_end","overlap"])

# print(f"✅ {len(results_df)} predicted exons processed.\n")
# print("Predicted exon counts per Ensembl ID:")
# print(dict(exon_counts))               # print but *not* added to DataFrame

# results_df



# # ╔════════════════════════════════════════════════════════════════════╗
# # ║  Poison‑exon overlap + per‑ID exon counts                          ║
# # ╚════════════════════════════════════════════════════════════════════╝
# import csv, h5py, numpy as np, pandas as pd, re
# from collections import defaultdict

# # ──────────────────────────────────────────────────────────────────────
# # ①  Paths / options
# # ──────────────────────────────────────────────────────────────────────
# POISON_TSV   = "SupplementaryDataFile1 - ST1.tsv"
# GENE_MAP_CSV = "Hs-hg38_genes_names.csv"

# PRED_FILES   = [
#     "/home/jovyan/shares/SR003.nfs2/caduseus_artem/neurips_rebbutal/pred/caduceus_ps-250k_shawerma-again-hg38_genome_poison-Hs.hdf5"
# ]

# # PRED_FILES   = [
# #     "/home/jovyan/shares/SR003.nfs2/caduseus_artem/neurips_rebbutal/pred/caduceus_ps-250k_shawerma-hg38-genes-Hs-chr20.hdf5",
# #     "/home/jovyan/dnalm/downstream_tasks/annotation/poison_exons/caduceus_ps_250k_shawerma_again_hg38_chr8_chr21_poison_Hs_chr21.hdf5"
# # ]
# # PRED_FILES   = [
# #     "/home/jovyan/dnalm/downstream_tasks/annotation/poison_exons/gena_best-1M-again-hg38_chr8_chr21_poison-Hs.hdf5",
# #     "/home/jovyan/shares/SR003.nfs2/caduseus_artem/neurips_rebbutal/pred/genatator_again-hg38-Hs_chr20.hdf5"
# # ]
# COORD_BASE   = 1              # 0 = BED half‑open ; 1 = GTF closed
# # KEEP_CHRS    = {"chr8", "chr20", "chr21"}
# KEEP_CHRS    = {f'chr{i}' for i in range(1, 24)}


# # ──────────────────────────────────────────────────────────────────────
# # ②  Utilities
# # ──────────────────────────────────────────────────────────────────────
# coord_re = re.compile(r"^(chr[\w]+)_(\d+)_(\d+)(?:_[\+\-])?$")

# def parse_coord(txt):
#     m = coord_re.match(txt)
#     return (m.group(1), int(m.group(2)), int(m.group(3))) if m else None

# def contiguous_ones(arr):
#     idx = np.where(arr == 1)[0]
#     if idx.size == 0: return []
#     split = np.where(np.diff(idx) > 1)[0] + 1
#     return [(seg[0], seg[-1] + 1) for seg in np.split(idx, split)]

# def overlap_kind(a0,a1,b0,b1):
#     if a1 <= b0 or a0 >= b1:            return "none"
#     if a0 <= b0 and a1 >= b1:           return "full"
#     return "partial"

# # ──────────────────────────────────────────────────────────────────────
# # ③  Gene‑symbol ↔ Ensembl map
# # ──────────────────────────────────────────────────────────────────────
# ensg2sym = {}
# with open(GENE_MAP_CSV) as f:
#     for row in csv.DictReader(f):
#         ensg2sym.setdefault(row["ID"].split(".")[0], row["gene_name"])

# # ──────────────────────────────────────────────────────────────────────
# # ④  Load poison exons  (keyed by Ensembl ID)
# #     NOTE: also capture 'transcript_biotype' from ST1.tsv
# # ──────────────────────────────────────────────────────────────────────
# gene2poison = defaultdict(list)
# with open(POISON_TSV) as f:
#     reader = csv.reader(f, delimiter="\t")
#     biotype_idx = None
#     first_row = True
#     for cols in reader:
#         if first_row:
#             # If header present, capture 'transcript_biotype' column index
#             if "transcript_biotype" in cols:
#                 biotype_idx = cols.index("transcript_biotype")
#                 first_row = False
#                 continue  # skip header row
#             first_row = False
#         if len(cols) < 2: continue
#         parsed = parse_coord(cols[0])
#         if parsed is None: continue
#         chrom, start, end = parsed
#         if chrom not in KEEP_CHRS: continue
#         gene_sym = cols[1]
#         # get biotype if available; ensure non-None string
#         transcript_biotype = "UNKNOWN"
#         if biotype_idx is not None and biotype_idx < len(cols):
#             val = cols[biotype_idx]
#             transcript_biotype = val if (val is not None and val != "") else "UNKNOWN"
#         # try to get ENSG from the same row or mapping table
#         ensg = next((c.split(".")[0] for c in cols[2:] if c.startswith("ENSG")),
#                     None) or next((k for k,v in ensg2sym.items() if v == gene_sym), None)
#         if ensg:
#             gene2poison[ensg].append({
#                 "chrom": chrom, "start": start, "end": end,
#                 "symbol": gene_sym,
#                 "transcript_biotype": transcript_biotype
#             })
#             ensg2sym.setdefault(ensg, gene_sym)

# # ──────────────────────────────────────────────────────────────────────
# # ⑤  Scan HDF5 files → predicted exon segments
# # ──────────────────────────────────────────────────────────────────────
# def is_tx(g):
#     k = g.keys()
#     return {"gene_predictions_forward","gene_predictions_reverse","gene_coordinates"}.issubset(k) \
#            and "ID" in g.attrs

# def walk(g):
#     yield g
#     for v in g.values():
#         if isinstance(v, h5py.Group):
#             yield from walk(v)

# pred_rows = []
# exon_counts = defaultdict(int)          # NEW: exon count per Ensembl ID

# for path in PRED_FILES:
#     with h5py.File(path, "r") as hf:
#         for grp in walk(hf):
#             if not is_tx(grp): continue
#             gid = grp.attrs["ID"]
#             gid = (gid.decode() if isinstance(gid, bytes) else gid).split(".")[0]
#             if gid not in gene2poison:   continue

#             chrom = grp.attrs["chromosome"]
#             chrom = chrom.decode() if isinstance(chrom, bytes) else chrom
#             tx_start, tx_end = map(int, grp["gene_coordinates"][:])

#             avg = 0.5*(grp["gene_predictions_forward"][:] + grp["gene_predictions_reverse"][:])  # 5×N
#             exon_mask = np.argmax(avg[[2,1],:], axis=0) == 1
#             for s,e in contiguous_ones(exon_mask):
#                 exon_counts[gid] += 1                     # count for this ID
#                 g0, g1 = tx_start + s, tx_start + e
#                 if COORD_BASE == 1: g0 += 1
#                 pred_rows.append({"gene_id": gid, "chrom": chrom,
#                                   "pred_start": g0, "pred_end": g1})

# # ──────────────────────────────────────────────────────────────────────
# # ⑥  Attach gene_symbol + one poison exon per row
# # ──────────────────────────────────────────────────────────────────────
# def choose_poison(ensg, chrom, mid):
#     poisons = gene2poison[ensg]
#     same = [p for p in poisons if p["chrom"] == chrom] or poisons
#     return min(same, key=lambda p: abs((p["start"]+p["end"])/2 - mid))

# full_rows = []
# for r in pred_rows:
#     gid, chrom = r["gene_id"], r["chrom"]
#     mid = (r["pred_start"] + (r["pred_end"] if COORD_BASE else r["pred_end"]-1))/2
#     pe = choose_poison(gid, chrom, mid)
#     a0, a1 = r["pred_start"], r["pred_end"] if COORD_BASE else r["pred_end"]-1
#     b0, b1 = pe["start"], pe["end"] if COORD_BASE else pe["end"]-1
#     r.update({"gene_symbol": ensg2sym.get(gid),
#               "poison_start": pe["start"],
#               "poison_end":   pe["end"],
#               "overlap":      overlap_kind(a0,a1,b0,b1),
#               "transcript_biotype": pe.get("transcript_biotype", "UNKNOWN")})
#     full_rows.append(r)

# # ──────────────────────────────────────────────────────────────────────
# # ⑦  Final DataFrame and exon‑count print‑out
# # ──────────────────────────────────────────────────────────────────────
# results_df = pd.DataFrame(full_rows,
#     columns=["gene_id","gene_symbol","chrom","pred_start","pred_end",
#              "poison_start","poison_end","overlap","transcript_biotype"])

# print(f"✅ {len(results_df)} predicted exons processed.\n")
# print("Predicted exon counts per Ensembl ID:")
# print(dict(exon_counts))               # print but *not* added to DataFrame

# results_df









# ╔════════════════════════════════════════════════════════════════════╗
# ║  Poison‑exon overlap + per‑ID exon counts                          ║
# ╚════════════════════════════════════════════════════════════════════╝
import csv, h5py, numpy as np, pandas as pd, re
from collections import defaultdict

# ──────────────────────────────────────────────────────────────────────
# ①  Paths / options
# ──────────────────────────────────────────────────────────────────────
POISON_TSV   = "SupplementaryDataFile1 - ST1.tsv"
GENE_MAP_CSV = "Hs-hg38_genes_names.csv"

PRED_FILES   = [
    "/home/jovyan/shares/SR003.nfs2/caduseus_artem/neurips_rebbutal/pred/gena_best-1M-again-hg38_genome_poison-Hs.hdf5"
]

COORD_BASE   = 1              # 0 = BED half‑open ; 1 = GTF closed
# IMPORTANT: include chrX/chrY in addition to autosomes
KEEP_CHRS    = {f"chr{i}" for i in range(1, 23)} | {"chrX", "chrY"}

# ──────────────────────────────────────────────────────────────────────
# ②  Utilities
# ──────────────────────────────────────────────────────────────────────
# Parse strings like 'chr3_183988293_183988359_-' (strand captured)
coord_re = re.compile(r"^(chr[\w]+)_(\d+)_(\d+)_(\+|\-)$")

def parse_coord(txt):
    m = coord_re.match(txt)
    return (m.group(1), int(m.group(2)), int(m.group(3)), m.group(4)) if m else None

def contiguous_ones(arr):
    idx = np.where(arr == 1)[0]
    if idx.size == 0: return []
    split = np.where(np.diff(idx) > 1)[0] + 1
    return [(seg[0], seg[-1] + 1) for seg in np.split(idx, split)]

def overlap_kind(a0,a1,b0,b1):
    if a1 <= b0 or a0 >= b1:            return "none"
    if a0 <= b0 and a1 >= b1:           return "full"
    return "partial"

# ──────────────────────────────────────────────────────────────────────
# ③  Gene‑symbol ↔ Ensembl map
# ──────────────────────────────────────────────────────────────────────
ensg2sym = {}
with open(GENE_MAP_CSV) as f:
    for row in csv.DictReader(f):
        ensg2sym.setdefault(row["ID"].split(".")[0], row["gene_name"])

# ──────────────────────────────────────────────────────────────────────
# ④  Load poison exons  (keyed by Ensembl ID)
#     NOTE: also capture 'transcript_biotype' and 'strand' from ST1.tsv
# ──────────────────────────────────────────────────────────────────────
gene2poison = defaultdict(list)
_expected_from_st1 = set()  # unique (chrom,start,end) directly from ST1 after KEEP_CHRS

with open(POISON_TSV) as f:
    reader = csv.reader(f, delimiter="\t")
    biotype_idx = None
    first_row = True
    for cols in reader:
        if first_row:
            if "transcript_biotype" in cols:
                biotype_idx = cols.index("transcript_biotype")
                first_row = False
                continue  # skip header row
            first_row = False

        if len(cols) < 2:
            continue

        parsed = parse_coord(cols[0])
        if parsed is None:
            continue
        chrom, start, end, strand = parsed
        if chrom not in KEEP_CHRS:
            continue

        _expected_from_st1.add((chrom, start, end))  # we check against 394 later

        gene_sym = cols[1]
        transcript_biotype = "UNKNOWN"
        if biotype_idx is not None and biotype_idx < len(cols):
            val = cols[biotype_idx]
            transcript_biotype = val if (val is not None and val != "") else "UNKNOWN"

        # try to get ENSG from the same row or mapping table
        ensg = next((c.split(".")[0] for c in cols[2:] if c.startswith("ENSG")),
                    None) or next((k for k,v in ensg2sym.items() if v == gene_sym), None)
        if ensg:
            gene2poison[ensg].append({
                "chrom": chrom, "start": start, "end": end,
                "symbol": gene_sym,
                "transcript_biotype": transcript_biotype,
                "strand": strand
            })
            ensg2sym.setdefault(ensg, gene_sym)

# Assert: the ST1 unique coordinate triples after KEEP_CHRS must be exactly 394
assert len(_expected_from_st1) == 394, (
    f"ST1 unique (chrom,start,end) after KEEP_CHRS == {len(_expected_from_st1)}, expected 394."
)

# Also assert that mapping to Ensembl didn't drop any ST1 exons
_expected_from_mapping = {(p["chrom"], p["start"], p["end"])
                          for ensg in gene2poison for p in gene2poison[ensg]}
assert _expected_from_mapping == _expected_from_st1, (
    "Mismatch between ST1 exons and Ensembl-mapped exons. "
    f"Lost {len(_expected_from_st1 - _expected_from_mapping)} entries "
    f"(e.g. {list((_expected_from_st1 - _expected_from_mapping))[:5]})."
)

# ──────────────────────────────────────────────────────────────────────
# ⑤  Scan HDF5 files → predicted exon segments
# ──────────────────────────────────────────────────────────────────────
def is_tx(g):
    k = g.keys()
    return {"gene_predictions_forward","gene_predictions_reverse","gene_coordinates"}.issubset(k) \
           and "ID" in g.attrs

def walk(g):
    yield g
    for v in g.values():
        if isinstance(v, h5py.Group):
            yield from walk(v)

pred_rows = []
exon_counts = defaultdict(int)          # exon count per Ensembl ID

for path in PRED_FILES:
    with h5py.File(path, "r") as hf:
        for grp in walk(hf):
            if not is_tx(grp): continue
            gid = grp.attrs["ID"]
            gid = (gid.decode() if isinstance(gid, bytes) else gid).split(".")[0]
            if gid not in gene2poison:   continue

            chrom = grp.attrs["chromosome"]
            chrom = chrom.decode() if isinstance(chrom, bytes) else chrom
            tx_start, tx_end = map(int, grp["gene_coordinates"][:])

            avg = 0.5*(grp["gene_predictions_forward"][:] + grp["gene_predictions_reverse"][:])  # 5×N
            exon_mask = np.argmax(avg[[2,1],:], axis=0) == 1
            for s,e in contiguous_ones(exon_mask):
                exon_counts[gid] += 1
                g0, g1 = tx_start + s, tx_start + e
                if COORD_BASE == 1: g0 += 1
                pred_rows.append({"gene_id": gid, "chrom": chrom,
                                  "pred_start": g0, "pred_end": g1})

# ──────────────────────────────────────────────────────────────────────
# ⑥  Attach gene_symbol + one poison exon per row
# ──────────────────────────────────────────────────────────────────────
def choose_poison(ensg, chrom, mid):
    poisons = gene2poison[ensg]
    same = [p for p in poisons if p["chrom"] == chrom] or poisons
    return min(same, key=lambda p: abs((p["start"]+p["end"])/2 - mid))

full_rows = []
for r in pred_rows:
    gid, chrom = r["gene_id"], r["chrom"]
    mid = (r["pred_start"] + (r["pred_end"] if COORD_BASE else r["pred_end"]-1))/2
    pe = choose_poison(gid, chrom, mid)
    a0, a1 = r["pred_start"], r["pred_end"] if COORD_BASE else r["pred_end"]-1
    b0, b1 = pe["start"], pe["end"] if COORD_BASE else pe["end"]-1
    r.update({"gene_symbol": ensg2sym.get(gid),
              "poison_start": pe["start"],
              "poison_end":   pe["end"],
              "overlap":      overlap_kind(a0,a1,b0,b1),
              "transcript_biotype": pe.get("transcript_biotype", "UNKNOWN")})
    full_rows.append(r)

# ──────────────────────────────────────────────────────────────────────
# ⑥b  Ensure every poison exon triple from ST1 appears at least once
# ──────────────────────────────────────────────────────────────────────
_seen_poison = {(r["chrom"], r["poison_start"], r["poison_end"]) for r in full_rows}
missing = _expected_from_st1 - _seen_poison

if missing:
    for ensg, plist in gene2poison.items():
        for p in plist:
            key = (p["chrom"], p["start"], p["end"])
            if key in missing:
                full_rows.append({
                    "gene_id": ensg,
                    "gene_symbol": ensg2sym.get(ensg),
                    "chrom": p["chrom"],
                    "pred_start": np.nan,
                    "pred_end":   np.nan,
                    "poison_start": p["start"],
                    "poison_end":   p["end"],
                    "overlap": "none",
                    "transcript_biotype": p.get("transcript_biotype", "UNKNOWN")
                })

# Recompute and assert again: results_df must cover EXACTLY 394 unique exons
_seen_poison = {(r["chrom"], r["poison_start"], r["poison_end"]) for r in full_rows}
assert len(_seen_poison) == 394, f"results_df unique poison exons = {len(_seen_poison)}, expected 394."
assert _seen_poison == _expected_from_st1, (
    "results_df still misses some ST1 exons after augmentation. "
    f"Missing {len(_expected_from_st1 - _seen_poison)} "
    f"(e.g. {list((_expected_from_st1 - _seen_poison))[:5]})"
)

# ──────────────────────────────────────────────────────────────────────
# ⑦  Final DataFrame and exon‑count print‑out
# ──────────────────────────────────────────────────────────────────────
results_df = pd.DataFrame(full_rows,
    columns=["gene_id","gene_symbol","chrom","pred_start","pred_end",
             "poison_start","poison_end","overlap","transcript_biotype"])

print(f"✅ {len(results_df)} predicted exons processed.\n")
print("Predicted exon counts per Ensembl ID:")
print(dict(exon_counts))               # print but *not* added to DataFrame

# Final sanity assert for the user's check:
# results_df[["chrom","poison_start","poison_end"]].drop_duplicates() must be 394 rows
_unique_count = results_df[["chrom","poison_start","poison_end"]].drop_duplicates().shape[0]
assert _unique_count == 394, f"Unique poison (chrom,start,end) in results_df is {_unique_count}, expected 394."

results_df



✅ 8907 predicted exons processed.

Predicted exon counts per Ensembl ID:
{'ENSG00000188157': 38, 'ENSG00000130939': 30, 'ENSG00000010803': 18, 'ENSG00000163312': 20, 'ENSG00000109320': 25, 'ENSG00000145348': 26, 'ENSG00000205403': 14, 'ENSG00000138709': 27, 'ENSG00000164169': 12, 'ENSG00000129116': 34, 'ENSG00000154556': 45, 'ENSG00000083857': 30, 'ENSG00000164151': 21, 'ENSG00000070759': 12, 'ENSG00000112902': 38, 'ENSG00000164190': 48, 'ENSG00000130449': 18, 'ENSG00000113163': 33, 'ENSG00000132842': 30, 'ENSG00000164329': 16, 'ENSG00000038427': 16, 'ENSG00000175471': 25, 'ENSG00000145730': 28, 'ENSG00000146021': 19, 'ENSG00000269113': 8, 'ENSG00000131503': 41, 'ENSG00000173261': 4, 'ENSG00000145868': 24, 'ENSG00000168246': 3, 'ENSG00000170085': 10, 'ENSG00000165671': 24, 'ENSG00000027847': 6, 'ENSG00000112033': 8, 'ENSG00000096060': 12, 'ENSG00000096070': 13, 'ENSG00000162374': 64, 'ENSG00000065491': 15, 'ENSG00000124688': 5, 'ENSG00000096401': 17, 'ENSG00000079841': 57, 'ENSG0000014

,gene_id,gene_symbol,chrom,pred_start,pred_end,poison_start,poison_end,overlap,transcript_biotype
0,ENSG00000188157,AGRN,chr1,1020120.0,1020373.0,1042691,1042748,none,poison
1,ENSG00000188157,AGRN,chr1,1022201.0,1022462.0,1042691,1042748,none,poison
2,ENSG00000188157,AGRN,chr1,1035277.0,1035324.0,1042691,1042748,none,poison
3,ENSG00000188157,AGRN,chr1,1040665.0,1040880.0,1042691,1042748,none,poison
4,ENSG00000188157,AGRN,chr1,1041173.0,1041397.0,1042691,1042748,none,poison
...,...,...,...,...,...,...,...,...,...
8902,ENSG00000083896,YTHDC1,chr4,68337572.0,68337900.0,68321372,68322358,none,poison
8903,ENSG00000083896,YTHDC1,chr4,68338283.0,68338384.0,68321372,68322358,none,poison
8904,ENSG00000083896,YTHDC1,chr4,68349726.0,68350090.0,68321372,68322358,none,poison
8905,ENSG00000163312,HELQ,chr4,NaN,NaN,83423358,83423448,none,poison


In [38]:
np.unique(results_df.overlap.to_numpy(), return_counts=True)

(array(['full', 'none', 'partial'], dtype=object), array([  35, 8850,   22]))

In [6]:
results_df[results_df["pred_start"].isna()]

,gene_id,gene_symbol,chrom,pred_start,pred_end,poison_start,poison_end,overlap,transcript_biotype
8905,ENSG00000163312,HELQ,chr4,NaN,NaN,83423358,83423448,none,poison
8906,ENSG00000119509,INVS,chr9,NaN,NaN,100251968,100252073,none,poison


In [3]:
len(results_df.gene_symbol.unique()) # why not 394?

349

In [40]:
# select only the poison_start and poison_end columns
unique_poison_coords = results_df[["chrom", "poison_start", "poison_end"]].drop_duplicates()

# now unique_poison_coords is a DataFrame with duplicate rows removed
print(unique_poison_coords.shape)

(394, 3)


In [41]:
# check exact-coordinate matches only among rows with full overlap
df = results_df[results_df["overlap"] == "full"].copy()

# be robust to object dtypes
for c in ["pred_start", "poison_start", "pred_end", "poison_end"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

n = len(df)
start_eq_mask = df["pred_start"] == df["poison_start"]
end_eq_mask   = df["pred_end"]   == df["poison_end"]

start_equal = int(start_eq_mask.sum())
end_equal   = int(end_eq_mask.sum())

print(f"Rows (overlap='full'): {n}")
print(f"pred_start == poison_start: {start_equal}/{n}")
print(f"pred_end   == poison_end  : {end_equal}/{n}")

# Optional: show first few mismatches
if start_equal != n:
    print("First start mismatches:")
    print(df.loc[~start_eq_mask, ["pred_start", "poison_start"]].head())

if end_equal != n:
    print("First end mismatches:")
    print(df.loc[~end_eq_mask, ["pred_end", "poison_end"]].head())

# NEW: return a DataFrame without mismatches (both start and end must match)
df_without_mismatches = df[start_eq_mask & end_eq_mask].copy()


Rows (overlap='full'): 35
pred_start == poison_start: 34/35
pred_end   == poison_end  : 34/35
First start mismatches:
      pred_start  poison_start
2728  93522942.0      93522946
First end mismatches:
       pred_end  poison_end
856  43635578.0    43635567


In [42]:
np.unique(df_without_mismatches.transcript_biotype.to_numpy(), return_counts=True)

(array(['poison', 'protein-coding', 'unknown'], dtype=object),
 array([22,  7,  4]))

In [43]:
results_df[results_df.overlap == 'full']

,gene_id,gene_symbol,chrom,pred_start,pred_end,poison_start,poison_end,overlap,transcript_biotype
178,ENSG00000138709,LARP1B,chr4,128092777.0,128093031.0,128092777,128093031,full,poison
446,ENSG00000130449,ZSWIM6,chr5,61525016.0,61525147.0,61525016,61525147,full,poison
686,ENSG00000145868,FBXO38,chr5,148411080.0,148411238.0,148411080,148411238,full,poison
856,ENSG00000124688,MAD2L1BP,chr6,43635487.0,43635578.0,43635487,43635567,full,unknown
874,ENSG00000096401,CDC5L,chr6,44432353.0,44432555.0,44432353,44432555,full,poison
942,ENSG00000112218,GPR63,chr6,96836265.0,96836377.0,96836265,96836377,full,protein-coding
1248,ENSG00000170027,YWHAG,chr7,76356905.0,76357030.0,76356905,76357030,full,poison
1313,ENSG00000146963,LUC7L2,chr7,139363176.0,139363259.0,139363176,139363259,full,poison
1655,ENSG00000164983,TMEM65,chr8,124351038.0,124351118.0,124351038,124351118,full,protein-coding
2229,ENSG00000175764,TTLL11,chr9,122040374.0,122040479.0,122040374,122040479,full,poison


In [5]:
results_df[results_df.overlap == 'full']

,gene_id,gene_symbol,chrom,pred_start,pred_end,poison_start,poison_end,overlap
145,ENSG00000062598,ELMO2,chr20,46382206,46382241,46382206,46382241,full
394,ENSG00000164983,TMEM65,chr8,124351038,124351118,124351038,124351118,full


In [4]:
# Build a BED with poison exons from "SupplementaryDataFile1 - ST1.tsv"
# Assumes coordinate field like: chr20_123_456_+

INPUT_TSV = "SupplementaryDataFile1 - ST1.tsv"
OUTPUT_BED = "poison_exons.bed"

# If your TSV coordinates are 1-based CLOSED (typical), keep this True.
# BED is 0-based HALF-OPEN, so we shift start by -1 and keep end as-is.
INPUT_IS_1BASED_CLOSED = True

count = 0
with open(INPUT_TSV, "r", encoding="utf-8") as fin, open(OUTPUT_BED, "w", encoding="utf-8") as fout:
    for line in fin:
        if not line.strip():
            continue
        cols = line.rstrip("\n").split("\t")
        coord = cols[0]
        parts = coord.split("_")
        if len(parts) < 3:
            # malformed line; skip
            continue
        chrom, start_s, end_s = parts[0], parts[1], parts[2]
        try:
            start_i = int(start_s)
            end_i = int(end_s)
        except ValueError:
            continue

        if INPUT_IS_1BASED_CLOSED:
            bed_start = start_i - 1
            bed_end = end_i
        else:
            # If your TSV were already 0-based half-open, you'd use:
            bed_start = start_i
            bed_end = end_i

        if bed_start < 0:
            bed_start = 0  # safety

        fout.write(f"{chrom}\t{bed_start}\t{bed_end}\n")
        count += 1

print(f"✅ Wrote {count} BED rows to {OUTPUT_BED}")


✅ Wrote 394 BED rows to poison_exons.bed


In [20]:
import pandas as pd
import numpy as np

TSV = "SupplementaryDataFile1 - ST1.tsv"

# read the TSV and locate the transcript_biotype column (robust to spacing/case)
df = pd.read_csv(TSV, sep="\t", dtype=str)
col = next(c for c in df.columns if c.strip().lower() == "transcript_biotype")

# get values as numpy array; normalize blanks to "unknown"
vals = df[col].astype(str)#.str.strip().replace("", "unknown").to_numpy()

# unique values and their counts
unique_biotypes, counts = np.unique(vals, return_counts=True)

# optional: pretty print
for b, n in zip(unique_biotypes, counts):
    print(f"{b}\t{n}")

# returned value in a notebook (tuple of arrays)
unique_biotypes, counts


poison	286
protein-coding	81
unknown	27


(array(['poison', 'protein-coding', 'unknown'], dtype=object),
 array([286,  81,  27]))

In [ ]:
# select only the poison_start and poison_end columns
unique_poison_coords = results_df[["poison_start", "poison_end"]].drop_duplicates()

# now unique_poison_coords is a DataFrame with duplicate rows removed
print(unique_poison_coords.shape)